# Download dos dados (Base dos Dados / BigQuery)

Notebook de apoio do projeto *Gênero e representação política*. Ele **só baixa os dados** e os salva em `data/` (parquet).
A limpeza está em `01_limpeza.ipynb` e a análise em `artigo.ipynb`.

**Como usar:** rode as células em ordem. Cada download só consulta o BigQuery se o arquivo ainda não existir em `data/`,
então rodar de novo não gasta cota. Para refazer um download, use `forcar=True`.

**Pré-requisitos:** `pip install basedosdados pandas pyarrow` e um projeto no Google Cloud.

## 1. Configuração

In [ ]:
import json
import datetime as dt
from pathlib import Path

import pandas as pd
import basedosdados as bd

BILLING = "a1-aedv"          # ID do projeto no Google Cloud

# Este notebook fica em notebooks/; as pastas abaixo ficam na raiz do repositório
DATA = Path("../data")
QUERIES = Path("../queries")
DATA.mkdir(exist_ok=True)
QUERIES.mkdir(exist_ok=True)

## 2. Função auxiliar

`baixar(nome, query)` executa a consulta, salva o resultado em `data/<nome>.parquet`, guarda a query em `queries/<nome>.sql`
(reprodutibilidade) e registra data do download, número de linhas e colunas em `data/<nome>_meta.json`
(útil para a seção de descrição dos dados do artigo).

In [ ]:
def baixar(nome, query, forcar=False):
    arq = DATA / f"{nome}.parquet"
    if arq.exists() and not forcar:
        print(f"{arq} já existe, lendo do disco (use forcar=True para baixar de novo).")
        return pd.read_parquet(arq)

    (QUERIES / f"{nome}.sql").write_text(query.strip() + "\n", encoding="utf-8")
    df = bd.read_sql(query, billing_project_id=BILLING)

    # colunas de data do BigQuery vêm como 'dbdate'; converte para datetime comum
    for c in df.columns:
        if str(df[c].dtype) == "dbdate":
            df[c] = pd.to_datetime(df[c])

    df.to_parquet(arq)
    meta = {
        "tabela": nome,
        "baixado_em": dt.datetime.now().isoformat(timespec="seconds"),
        "linhas": int(len(df)),
        "colunas": list(df.columns),
    }
    (DATA / f"{nome}_meta.json").write_text(
        json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"Salvo: {arq} ({len(df):,} linhas, {df.shape[1]} colunas)")
    return df


def colunas(tabela, dataset="br_tse_eleicoes"):
    """Lista as colunas de uma tabela da Base dos Dados (consulta de metadados)."""
    q = f"""
    SELECT column_name, data_type
    FROM `basedosdados.{dataset}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{tabela}'
    ORDER BY ordinal_position
    """
    return bd.read_sql(q, billing_project_id=BILLING)

## 3. Explorar: quais tabelas e colunas existem?

Rode antes de escrever qualquer query nova. **Confira os nomes reais**, porque o schema pode mudar.

In [ ]:
tabelas = bd.read_sql(
    """
    SELECT table_name
    FROM `basedosdados.br_tse_eleicoes.INFORMATION_SCHEMA.TABLES`
    ORDER BY table_name
    """,
    billing_project_id=BILLING,
)
tabelas

In [ ]:
# Descomente para ver as colunas de cada tabela de interesse
# colunas("candidatos")
# colunas("resultados_candidato")
# colunas("despesas_candidato")
# colunas("receitas_candidato")

## 4. Candidaturas (perguntas 1, 2, 3 e 5)

Candidaturas de 2002 em diante, com nomes de UF e município vindos dos diretórios da Base dos Dados.
Todas as colunas são selecionadas explicitamente (nada de `SELECT *`).

In [ ]:
query_candidatos = """
SELECT
    dados.ano as ano,
    dados.id_eleicao as id_eleicao,
    dados.tipo_eleicao as tipo_eleicao,
    dados.data_eleicao as data_eleicao,
    dados.sigla_uf as sigla_uf,
    diretorio_sigla_uf.nome as sigla_uf_nome,
    dados.id_municipio as id_municipio,
    diretorio_id_municipio.nome as id_municipio_nome,
    dados.id_municipio_tse as id_municipio_tse,
    diretorio_id_municipio_tse.nome as id_municipio_tse_nome,
    dados.numero as numero,
    dados.nome as nome,
    dados.nome_urna as nome_urna,
    dados.numero_partido as numero_partido,
    dados.sigla_partido as sigla_partido,
    dados.cargo as cargo,
    dados.situacao as situacao,
    dados.idade as idade,
    dados.genero as genero,
    dados.instrucao as instrucao,
    dados.ocupacao as ocupacao,
    dados.sigla_uf_nascimento as sigla_uf_nascimento,
    dados.raca as raca
FROM `basedosdados.br_tse_eleicoes.candidatos` AS dados
LEFT JOIN (SELECT DISTINCT sigla, nome FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
LEFT JOIN (SELECT DISTINCT id_municipio, nome FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
LEFT JOIN (SELECT DISTINCT id_municipio_tse, nome FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio_tse
    ON dados.id_municipio_tse = diretorio_id_municipio_tse.id_municipio_tse
WHERE dados.ano >= 2002
"""

candidatos = baixar("candidatos", query_candidatos)
candidatos.shape

## 5. Resultados: quem foi eleito (pergunta 2)

**Atenção:** a tabela de candidaturas acima não traz a informação de eleito/não eleito. Ela vem da tabela de resultados.
Antes de escrever a query:

1. Rode `colunas("resultados_candidato")` na seção 3.
2. Veja se existe uma coluna de identificação do candidato (como um sequencial) que também esteja em `candidatos`.
   Se existir só na tabela de resultados, inclua-a também na query de candidatos da seção 4 e baixe de novo com `forcar=True`,
   para poder cruzar as duas.
3. Preencha a query abaixo com os nomes reais.

In [ ]:
# query_resultados = """
# SELECT
#     ano, tipo_eleicao, sigla_uf, id_municipio, cargo,
#     -- <colunas de identificação do candidato>,
#     -- <coluna de resultado (eleito / não eleito)>,
#     -- <coluna de votos>
# FROM `basedosdados.br_tse_eleicoes.resultados_candidato`
# WHERE ano >= 2002
# """
# resultados = baixar("resultados_candidato", query_resultados)

## 6. Financiamento de campanha (pergunta 4)

As tabelas de receitas e despesas são grandes e só são consistentes a partir de 2014. **Agregue no SQL** (por exemplo, por ano,
cargo, partido e candidato) em vez de baixar linha por linha, e teste primeiro com um único ano. Estime os GB processados
no console do BigQuery antes de rodar.

In [ ]:
# query_receitas = """
# SELECT
#     ano, cargo, sigla_partido,
#     -- <coluna de identificação do candidato>,
#     SUM(<coluna de valor>) AS receita_total
# FROM `basedosdados.br_tse_eleicoes.receitas_candidato`
# WHERE ano = 2022
# GROUP BY ano, cargo, sigla_partido  -- + identificação do candidato
# """
# receitas_2022 = baixar("receitas_candidato_2022", query_receitas)

## 7. Checagem final

In [ ]:
for arq in sorted(DATA.glob("*.parquet")):
    print(f"{arq.name:45s} {arq.stat().st_size / 1e6:8.1f} MB")

candidatos["ano"].value_counts().sort_index()